In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Chargement du dataset AVEC Nutri-Score (notre base d'apprentissage)
# Remarque : j'ai mis sep='\t' (tabulation) au cas où il vienne du même export OFF, 
# mais si c'est un CSV classique que tu as créé, remplace par sep=','
df_train = pd.read_csv('produits_avec_nutriscore.csv', sep='\t', dtype={'code': str}, low_memory=False)

# 2. Chargement du dataset SANS Nutri-Score (celui qu'on veut compléter)
df_predict = pd.read_csv('produits_sans_nutriscore.csv', sep='\t', dtype={'code': str}, low_memory=False)

# Affichage des dimensions pour vérifier que tout est bien chargé
print(f"Taille du dataset d'entraînement (AVEC Nutri-Score) : {df_train.shape}")
print(f"Taille du dataset à compléter (SANS Nutri-Score) : {df_predict.shape}")

Taille du dataset d'entraînement (AVEC Nutri-Score) : (59665, 1)
Taille du dataset à compléter (SANS Nutri-Score) : (33216, 1)


In [7]:
import io

# Liste des colonnes à garder
colonnes_utiles = [
    'code', 'product_name', 'nova_group', 'additives_n',
    'energy-kj_100g', 'energy-kcal_100g', 'sugars_100g', 'carbohydrates_100g',
    'saturated-fat_100g', 'fat_100g', 'salt_100g', 'proteins_100g', 'fiber_100g', 
    'fruits-vegetables-legumes_100g', 'nutriscore_grade' # On garde le grade pour le train
]

colonnes_nutriments = ['sugars_100g', 'carbohydrates_100g', 'saturated-fat_100g', 'fat_100g', 'salt_100g', 'proteins_100g', 'fiber_100g']

def corriger_dataframe_mal_sep(df_input):
    # Cas fréquent: CSV lu avec sep='\t' alors qu'il est séparé par des virgules
    if df_input.shape[1] == 1 and ',' in df_input.columns[0]:
        texte_csv = df_input.columns[0] + '\n' + '\n'.join(df_input.iloc[:, 0].astype(str).tolist())
        return pd.read_csv(io.StringIO(texte_csv), dtype={'code': str}, low_memory=False)
    return df_input

def preparer_donnees(df_input):
    # 0. Correction du parsing si nécessaire
    df_input = corriger_dataframe_mal_sep(df_input)

    # 1. Copie et filtrage des colonnes (en ignorant celles qui n'existent pas dans df_predict)
    cols_presentes = [c for c in colonnes_utiles if c in df_input.columns]
    df = df_input[cols_presentes].copy()
    
    # 2. Conversion des colonnes numériques utiles
    for col in ['nova_group', 'additives_n'] + colonnes_nutriments:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Suppression des lignes sans nom de produit
    df = df.dropna(subset=['product_name'])
    
    # 4. Plafonner à 100g et empêcher les valeurs négatives
    for col in colonnes_nutriments:
        if col in df.columns:
            df.loc[df[col] > 100, col] = 100
            df.loc[df[col] < 0, col] = 0
            
    # 5. Cohérence Sucres/Glucides
    if 'sugars_100g' in df.columns and 'carbohydrates_100g' in df.columns:
        masque_sucre = df['sugars_100g'] > df['carbohydrates_100g']
        df.loc[masque_sucre, 'sugars_100g'] = df.loc[masque_sucre, 'carbohydrates_100g']
        
    # 6. Supprimer les aberrations (Somme > 100g)
    cols_somme = ['carbohydrates_100g', 'fat_100g', 'proteins_100g', 'salt_100g']
    cols_somme_presentes = [c for c in cols_somme if c in df.columns]
    if cols_somme_presentes:
        somme_macros = df[cols_somme_presentes].fillna(0).sum(axis=1)
        df = df[somme_macros <= 100]
    
    return df

# Application de la fonction à nos deux datasets
df_train_clean = preparer_donnees(df_train)
df_predict_clean = preparer_donnees(df_predict)

# Pour l'entraînement, on s'assure de ne garder que des lignes où les nutriments cibles sont connus
colonnes_cibles_presentes = [c for c in colonnes_nutriments if c in df_train_clean.columns]
df_train_clean = df_train_clean.dropna(subset=colonnes_cibles_presentes)

print(f"Train propre : {df_train_clean.shape}")
print(f"Predict propre : {df_predict_clean.shape}")

Train propre : (58903, 15)
Predict propre : (32394, 15)


In [8]:
# Export des datasets traités en CSV
df_train_clean.to_csv('traité_avec_nutriscore.csv', index=False, encoding='utf-8-sig')
df_predict_clean.to_csv('traité_sans_nutriscore.csv', index=False, encoding='utf-8-sig')

print(f"✅ 'traité_avec_nutriscore.csv' sauvegardé ({len(df_train_clean)} lignes)")
print(f"✅ 'traité_sans_nutriscore.csv' sauvegardé ({len(df_predict_clean)} lignes)")

# supprime les fichiers d'origine pour éviter les confusions
import os
os.remove('produits_avec_nutriscore.csv')
os.remove('produits_sans_nutriscore.csv')
print("✅ Fichiers d'origine supprimés pour éviter les confusions")

✅ 'traité_avec_nutriscore.csv' sauvegardé (58903 lignes)
✅ 'traité_sans_nutriscore.csv' sauvegardé (32394 lignes)
✅ Fichiers d'origine supprimés pour éviter les confusions
